# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MansiNegi281/CapstoneFlyrankAI/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Pages are ranked by the model's predicted decline probability rather than
the Week 4 hand-written score, since Weeks 5-6 showed the model outperforms
the baseline rule on precision (even under the more honest grouped split).
Reason codes stay human-readable so a content editor can see why a page was
flagged, not just its rank number. This mirrors the FlyRank paper's own
approach of treating optimization flags as workflow cues rather than
absolute quality judgments (Myth #3 — flagged content often outperforms
zero-flag content because flags require enough data to trigger at all).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("/content/content_refresh_anonymized (1).csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

features = ["search_volume", "ctr", "avg_position", "engagement_rate",
            "scroll_rate", "content_age_days", "days_since_last_update",
            "impressions_90d", "sessions_90d"]

X = df[features].fillna(0)
y = df["is_declining"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

df["decline_probability"] = rf.predict_proba(X)[:, 1]

def reason_code(row):
    if row["trend_direction"] == "down":
        return "DECLINING_TRAFFIC"
    elif row["avg_position"] > 20:
        return "POOR_POSITION"
    elif row["days_since_last_update"] > 180:
        return "STALE_CONTENT"
    elif row["ctr"] < 0.05:
        return "LOW_CTR"
    else:
        return "MONITOR"

df["reason_code"] = df.apply(reason_code, axis=1)
df["action_label"] = "REFRESH_CONTENT"

ranked_queue = df.sort_values("decline_probability", ascending=False)
ranked_queue[["content_id", "decline_probability", "reason_code", "action_label"]].head(20)

,content_id,decline_probability,reason_code,action_label
12972,content_d5b833d82e72,1.0,DECLINING_TRAFFIC,REFRESH_CONTENT
2107,content_60e8eba31e41,1.0,DECLINING_TRAFFIC,REFRESH_CONTENT
25751,content_1d462e6151e1,1.0,DECLINING_TRAFFIC,REFRESH_CONTENT
2602,content_a2a9ec32d147,1.0,DECLINING_TRAFFIC,REFRESH_CONTENT
24470,content_a0f750ca1b2a,1.0,DECLINING_TRAFFIC,REFRESH_CONTENT
27851,content_2cf20c4ba4db,1.0,DECLINING_TRAFFIC,REFRESH_CONTENT
15488,content_8ee91fc16aa2,1.0,DECLINING_TRAFFIC,REFRESH_CONTENT
2944,content_30e28ce98183,1.0,DECLINING_TRAFFIC,REFRESH_CONTENT
6970,content_af8c97b7c3fc,1.0,DECLINING_TRAFFIC,REFRESH_CONTENT
8078,content_9903bd14be6c,1.0,DECLINING_TRAFFIC,REFRESH_CONTENT


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use: a content team lead uses this queue at the start of a review
cycle to decide which pages to look at first, out of far more pages than
they have time to manually audit.

Limits: this ranks candidates for review — it does not replace human
judgment on any single page. It stops being valid for content types or
clients meaningfully different from this dataset's mix, and for any page
younger than the 90-day window used to build its features. Per the research
paper's own freshness findings, a refresh recommendation is most reliable
for pages that were already substantive before going stale — thin pages
that were weak to begin with see a smaller benefit from refreshing (Myth #7,
"Freshness amplifies quality, it does not replace it").

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting, a person must check: whether the apparent decline is
seasonal, whether a recent site or tracking change explains a drop, and
whether the page still matches current search intent for its keyword.

Never automate: publishing content changes directly from this queue,
removing or de-indexing a page based on this score alone, or treating a
single high-probability page as certain to be declining without a human
look. The research paper's Myth #5 finding is a useful reminder here too —
don't assume a page's authorship (AI-assisted or not) explains its decline;
process and editing quality matter more than that binary.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Retrain if: precision on a fresh validation slice drops meaningfully below
the Week 6 grouped-split number, if the trend_direction distribution shifts
significantly from the current mix (down: 16262, stable: 5962, up: 4388,
new: 2236, flat: 1152), or once enough new content/clients are added that
the training window no longer represents current data.

Monitor: track 30/60/90-day impression and position changes on pages that
were acted on from this queue, similar to the "Measure" step in each of the
research paper's playbook recommendations — a recommendation is only useful
if its real-world outcome is checked afterward.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
os.makedirs("work/outputs", exist_ok=True)

ranked_queue[["content_id", "decline_probability", "reason_code", "action_label"]].to_csv(
    "work/outputs/model_action_playbook.csv", index=False
)
print("Saved playbook CSV — this feeds directly into your capstone's Results and Ranked recommendations sections")

Saved playbook CSV — this feeds directly into your capstone's Results and Ranked recommendations sections


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.